In [1]:
# Non-Linear GRANGER CAUSALITY
import sys 

sys.path.append("Nonlincausality-master") # to make python see the folder

import nonlincausality as nlc
import matplotlib.pyplot as plt
from nonlincausality.utils import prepare_data_for_prediction
import pandas as pd
import seaborn as sns

In [2]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

pkl_path = Path("data\processed_data\combined_latest_snapshot_20251212.pkl")

with open(pkl_path, "rb") as f:
    data = pickle.load(f)

print(type(data))
print(len(data))
print(data[0].keys())

item = data[0]

print(item["date"])
print(item["sic_image"].shape)
print(item["lat"].shape, item["lon"].shape)
print(item["env"].keys())
print(item["missing_env"])

# --- SORT DATASET TEMPORALLY ---
combined_dataset = sorted(data, key=lambda x: x["date"])

<class 'list'>
91
dict_keys(['date', 'sic_image', 'lat', 'lon', 'env', 'missing_env'])
2025-09-14 00:00:00+07:00
(448, 304)
(448, 304) (448, 304)
dict_keys(['u10', 'v10', 't2m', 'msl', 'sst', 'tp', 'mwd', 'mwp', 'swh'])
[]


In [3]:
item0 = combined_dataset[0]

print("Date:", item0["date"])
print("Missing env list:", item0["missing_env"])

for var in item0["env"]:
    print(var, type(item0["env"][var]))
    print(var, item0["env"][var].shape)

Date: 2025-09-14 00:00:00+07:00
Missing env list: []
u10 <class 'numpy.ndarray'>
u10 (448, 304)
v10 <class 'numpy.ndarray'>
v10 (448, 304)
t2m <class 'numpy.ndarray'>
t2m (448, 304)
msl <class 'numpy.ndarray'>
msl (448, 304)
sst <class 'numpy.ndarray'>
sst (448, 304)
tp <class 'numpy.ndarray'>
tp (448, 304)
mwd <class 'numpy.ndarray'>
mwd (448, 304)
mwp <class 'numpy.ndarray'>
mwp (448, 304)
swh <class 'numpy.ndarray'>
swh (448, 304)


In [4]:
# # --- target time series (SIC) ---
# sic_ts = np.array([
#     np.nanmean(item["sic_image"])
#     for item in combined_dataset
# ])

# # --- environmental time series ---
# env_vars = list(combined_dataset[0]["env"].keys())

# # keep only dates where ALL env vars are present
# filtered_dataset = []

# for item in combined_dataset:
#     if all(item["env"][var] is not None for var in env_vars):
#         filtered_dataset.append(item)

# print(f"Original length: {len(combined_dataset)}")
# print(f"Filtered length: {len(filtered_dataset)}")

# env_ts = {}

# for var in env_vars:
#     env_ts[var] = np.array([
#         np.nanmean(item["env"][var])
#         for item in filtered_dataset
#     ])

# print("Variables:", env_vars)
# print("sic_ts:", sic_ts.shape)
# for var in env_vars:
#     print(var, env_ts[var].shape, np.isnan(env_ts[var]).sum())

# # sic_ts.shape = (T,)
# # env_ts[var].shape = (T,)

In [5]:
env_vars = [
    "u10", "v10", "t2m", "msl", "sst",
    "tp", "mwd", "mwp", "swh"
]

filtered_dataset = [
    item for item in combined_dataset
    if item["sic_image"] is not None
    and all(item["env"].get(v) is not None for v in env_vars)
]

sic_ts = np.array([
    np.nanmean(item["sic_image"])
    for item in filtered_dataset
])

env_ts = {}

for v in env_vars:
    values = []
    
    for item in filtered_dataset:
        field = item["env"][v]
        
        if np.isnan(field).all():
            values.append(np.nan)
        else:
            values.append(np.nanmean(field))
            
    env_ts[v] = np.array(values)


In [6]:
results_by_var = {}

for var in env_vars:
    sic_ts = []
    env_var_ts = []

    for item in combined_dataset:
        if item["env"][var] is not None:
            sic_ts.append(np.nanmean(item["sic_image"]))
            env_var_ts.append(np.nanmean(item["env"][var]))

    sic_ts = np.array(sic_ts)
    env_var_ts = np.array(env_var_ts)

    if len(sic_ts) < 20:
        print(f"[SKIP] {var}: too few samples ({len(sic_ts)})")
        continue

    # data = np.vstack([sic_ts, env_var_ts]).T
    data = np.column_stack([sic_ts, env_var_ts])

    print(f"{var}: data shape = {data.shape}")

    # now safe to run nonlincausality
    results_by_var[var] = data


u10: data shape = (87, 2)
v10: data shape = (82, 2)
t2m: data shape = (82, 2)
msl: data shape = (82, 2)
sst: data shape = (82, 2)
tp: data shape = (82, 2)
mwd: data shape = (84, 2)
mwp: data shape = (84, 2)
swh: data shape = (84, 2)


In [7]:
# dropped_dates = [
#     item["date"] for item in combined_dataset
#     if item not in filtered_dataset
# ]

# print("Dropped dates due to missing ERA5:", dropped_dates)

In [8]:
print("Filtered dataset length:", len(filtered_dataset))

for var in env_vars:
    count_valid = sum(
        item["env"][var] is not None
        for item in combined_dataset
    )
    print(f"{var}: valid days = {count_valid}")


Filtered dataset length: 82
u10: valid days = 87
v10: valid days = 82
t2m: valid days = 82
msl: valid days = 82
sst: valid days = 82
tp: valid days = 82
mwd: valid days = 84
mwp: valid days = 84
swh: valid days = 84


In [9]:
valid_idx_by_var = {}

for var in env_vars:
    valid_idx = []

    for i, item in enumerate(combined_dataset):
        field = item["env"].get(var)
        if field is None:
            continue

        env_val = np.nanmean(field)
        sic_val = np.nanmean(item["sic_image"])

        if not np.isnan(env_val) and not np.isnan(sic_val):
            valid_idx.append(i)

    valid_idx_by_var[var] = set(valid_idx)


In [10]:
common_idx = set.intersection(*valid_idx_by_var.values())
common_idx = sorted(common_idx)

print("Common samples:", len(common_idx))


Common samples: 82


In [11]:
sic_ts = np.array([
    np.nanmean(combined_dataset[i]["sic_image"])
    for i in common_idx
])

env_ts = {
    var: np.array([
        np.nanmean(combined_dataset[i]["env"][var])
        for i in common_idx
    ])
    for var in env_vars
}


In [12]:
data = np.column_stack(
    [sic_ts] + [env_ts[v] for v in env_vars]
)

print(data.shape)

(82, 10)


In [13]:
data = np.vstack([
    sic_ts,          # X (target)
    env_ts["t2m"]    # Y (candidate cause)
]).T

print(data.shape)   # (T, 2)

(82, 2)


In [14]:
results = {}

for i, var in enumerate(env_vars, start=1):
    data_2d = np.column_stack([sic_ts, env_ts[var]])
    
    res = nlc(
        data_2d,
        maxlag = 10,
        x_test = [0], #sic
        y_test = [1] # env var
    )
    
    results[var] = res

TypeError: 'module' object is not callable

In [ ]:
T = data.shape[0]

train_end = int(0.6 * T)
val_end   = int(0.8 * T)

data_train = data[:train_end]
data_val   = data[train_end:val_end]
data_test  = data[val_end:]

In [ ]:
lags = [1, 3, 5, 7]   # daily lags (you can change)

results = nlc.nonlincausalityNN(
    x=data_train,
    maxlag=lags,
    NN_config=['d','dr','d','dr'],
    NN_neurons=[64, 0.1, 64, 0.1],
    x_test=data_test,
    x_val=data_val,
    run=3,
    epochs_num=[50, 50],
    learning_rate=[0.001, 0.0001],
    batch_size_num=32,
    verbose=True,
    plot=True
)

ValueError: maxlag should be < nobs

In [ ]:
summary = []

for lag in lags:
    res = results[lag]

    p_value = res.p_value
    test_stat = res._test_statistic

    print("="*50)
    print(f"Variable: t2m → SIC")
    print(f"Lag: {lag}")
    print(f"Test statistic: {test_stat:.4f}")
    print(f"p-value: {p_value:.4e}")
    print("Causal:", "YES" if p_value < 0.05 else "NO")

    summary.append({
        "variable": "t2m",
        "lag": lag,
        "p_value": p_value,
        "test_stat": test_stat
    })

    # loss curves
    nlc.plot_history_loss(
        res.best_history_X,
        res.best_history_XY
    )
    plt.title(f"t2m → SIC | lag={lag}")
    plt.show()

In [ ]:
all_results = []

for var in env_vars:
    data = np.vstack([sic_ts, env_ts[var]]).T

    data_train = data[:train_end]
    data_val   = data[train_end:val_end]
    data_test  = data[val_end:]

    results = nlc.nonlincausalityNN(
        x=data_train,
        maxlag=lags,
        NN_config=['d','dr','d','dr'],
        NN_neurons=[64, 0.1, 64, 0.1],
        x_test=data_test,
        x_val=data_val,
        run=3,
        epochs_num=[40, 40],
        learning_rate=[0.001, 0.0001],
        batch_size_num=32,
        verbose=False,
        plot=False
    )

    for lag in lags:
        res = results[lag]
        all_results.append({
            "variable": var,
            "lag": lag,
            "p_value": res.p_value,
            "test_stat": res._test_statistic,
            "causal": res.p_value < 0.05
        })


In [ ]:
df = pd.DataFrame(all_results)

# heatmap of p-values
pivot = df.pivot(index="variable", columns="lag", values="p_value")

plt.figure(figsize=(8,5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".2e",
    cmap="viridis_r"
)
plt.title("Nonlinear Granger causality p-values (ENV → SIC)")
plt.show()


In [ ]:
causal_vars = (
    df[df["causal"]]
    .groupby("variable")
    .agg(
        best_p=("p_value", "min"),
        best_lag=("lag", "min")
    )
)

print(causal_vars)
